# Fase 2: Compuerta Estocástica Híbrida (BFCL) y Tree Batching con Máscara Causal 2D (GPU T4)

Este notebook ejecuta de principio a fin la **Fase Experimental 2** del proyecto sobre GPU T4 (tarda solo **~3 a 4 minutos** en total porque el catálogo BFCL tiene 24 APIs y secuencias compactas):

1. **Verificación de Hardware y Suite Completa de Pruebas (`unittest`):** Comprueba las 23 pruebas unitarias y de integración (Fase 1 + Fase 2, incluyendo equivalencia matemática exacta de *Tree Batching* en Qwen).
2. **Entrenamiento LoRA en BFCL (24 APIs ejecutables) + Calibración L-BFGS + Compuerta Estocástica Híbrida ($\tau$):** Entrena el `DecisionModel` ($T=1$), calibra $T^*$, compara contra **ReAct ($T>1$)** y evalúa la compuerta $\tau \in [0.50, 0.95]$.
3. **Tree Batching en Vivo (Sección II-C del paper):** Mide en la GPU T4 la ganancia de latencia y ahorro de tokens al evaluar $M \in \{1, 2, 4, 8, 16\}$ preguntas concurrentes sobre un mismo tronco común (estado $s_t$ + catálogo) en **1 sola pasada con máscara causal 2D** frente a $M$ pasadas separadas.
4. **Visualización y Descarga (`phase2_results.zip`):** Muestra en pantalla y empaqueta las 3 figuras de publicación:
   - `hybrid_gate_tradeoff.png`
   - `bfcl_syntax_and_accuracy.png`
   - `tree_batching_latency.png`

## 1. Verificación del Hardware (GPU T4)

In [ ]:
!nvidia-smi

## 2. Clonar/Actualizar el Repositorio e Instalar Dependencias

In [ ]:
import os

if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")
    if not os.path.exists("PAM-F2"):
        !git clone https://github.com/PneumothoraxX/PAM-F2.git
    os.chdir("/kaggle/working/PAM-F2")
    !git pull origin main
elif not os.path.exists("src"):
    if not os.path.exists("PAM-F2"):
        !git clone https://github.com/PneumothoraxX/PAM-F2.git
    os.chdir("PAM-F2")
    !git pull origin main

print("Directorio activo:", os.getcwd())
!pip uninstall -y -q torchao
!pip install -q --upgrade pip
!pip install -q torch transformers peft accelerate datasets scipy scikit-learn matplotlib tqdm

## 3. Ejecutar la Suite Completa de 23 Pruebas

In [ ]:
import os
if os.path.exists("/kaggle/working/PAM-F2"):
    os.chdir("/kaggle/working/PAM-F2")

!PYTHONPATH=. python -m unittest discover -s tests -v

## 4. Ejecutar Fase 2 en Vivo: Entrenamiento BFCL, Compuerta Híbrida $\tau$, ReAct y Tree Batching 2D

Con `--run-live-model`, entrena y evalúa `Qwen/Qwen2.5-0.5B-Instruct` sobre las 24 APIs de BFCL (solo 60 pasos por época $\approx$ 2-3 minutos en T4) y mide *Tree Batching* en GPU.

In [ ]:
import os
if os.path.exists("/kaggle/working/PAM-F2"):
    os.chdir("/kaggle/working/PAM-F2")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "results/phase2_bfcl"
EPOCHS = 3
BATCH_SIZE = 8
TAU = 0.75

!PYTHONPATH=. python -m src.phase2_hybrid_bfcl.evaluate \
    --results-dir $OUTPUT_DIR \
    --model $MODEL_NAME \
    --run-live-model \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --tau $TAU

## 5. Visualizar Métricas y las 3 Figuras de Fase 2

In [ ]:
import json
import os
from pathlib import Path
from IPython.display import Image, display

if os.path.exists("/kaggle/working/PAM-F2"):
    os.chdir("/kaggle/working/PAM-F2")

OUTPUT_DIR = globals().get("OUTPUT_DIR", "results/phase2_bfcl")
res_dir = Path(OUTPUT_DIR)
metrics_path = res_dir / "phase2_metrics.json"

if metrics_path.exists():
    m = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("=== RESUMEN DE MÉTRICAS FASE 2 (BFCL + COMPUERTA + TREE BATCHING) ===")
    print(f"Temperatura Óptima L-BFGS (T*):       {m['optimal_temperature']:.4f}")
    print(f"Nivel 1 Aislado Accuracy:             {m['level1_isolated_accuracy'] * 100:.2f}%")
    print(f"Nivel 1 ECE (Antes -> Después):       {m['level1_ece_before']:.4f} -> {m['level1_ece_after']:.4f}")
    print(f"Agente Híbrido Accuracy (τ={m['recommended_tau']:.2f}):    {m['hybrid_accuracy_at_recommended_tau'] * 100:.2f}%")
    print(f"Despacho Directo Nivel 1 (τ={m['recommended_tau']:.2f}):   {m['hybrid_level1_coverage_at_recommended_tau'] * 100:.1f}%")
    print(f"Latencia Efectiva Media Híbrida:      {m['hybrid_effective_latency_ms_at_recommended_tau']:.1f} ms")
    tb16 = m["tree_batching_benchmark"][-1]
    print(f"Tree Batching (M={tb16['num_questions']}):                 {tb16['naive_latency_ms']:.1f} ms (Ingenuo) -> {tb16['tree_latency_ms']:.1f} ms (Tree, {tb16['speedup_factor']:.2f}x speedup, {tb16['token_reduction_ratio']:.2f}x menos tokens)")

for fig_name in [
    "hybrid_gate_tradeoff.png",
    "bfcl_syntax_and_accuracy.png",
    "tree_batching_latency.png",
]:
    fig_file = res_dir / fig_name
    if fig_file.exists():
        print(f"\n--- {fig_name} ---")
        display(Image(filename=str(fig_file)))

## 6. Empaquetar Resultados en `phase2_results.zip` para Descarga

In [ ]:
import os
import shutil

if os.path.exists("/kaggle/working/PAM-F2"):
    os.chdir("/kaggle/working/PAM-F2")

OUTPUT_DIR = globals().get("OUTPUT_DIR", "results/phase2_bfcl")
!zip -r phase2_results.zip $OUTPUT_DIR
if os.path.exists("/kaggle/working") and os.path.abspath("phase2_results.zip") != "/kaggle/working/phase2_results.zip":
    shutil.copy("phase2_results.zip", "/kaggle/working/phase2_results.zip")
!ls -lh phase2_results.zip